# Reasoning Analysis Example

This notebook demonstrates how to:
1. Extract reasoning from eval logs
2. Get summaries using an LLM
3. Cluster reasoning instances to find patterns

In [ ]:
# Import necessary modules
from inspect_ai.log import read_eval_log
import sys
sys.path.insert(0, '/workspace/rl-character/analysis/reasoning')

from reasoning import (
    Reasoning,
    Cluster,
    ReasoningProcessor,
    extract_reasoning_from_evallog,
    save_reasonings,
    load_reasonings,
    save_clusters
)

## Step 1: Extract Reasoning from Eval Log

Load an eval log and extract reasoning instances. Each Reasoning object contains:
- `content`: The model's generated reasoning
- `prompt`: The original prompt given to the model
- `sample_id`: Unique ID for the sample
- `epoch`: Epoch number
- `target`: Target label (e.g., 'hack' or 'clean')
- `score`: Score from the eval (e.g., 'C' for correct, 'I' for incorrect)
- `string_id`: Unique identifier combining sample_id and epoch

Note: The `idx` parameter specifies which generation/chat_history to extract.
For this specific eval setup, the indices correspond to:
- 0: quality_binary
- 1: user_binary
- 2: good_binary
- 3: prod_binary
- 4: impression_binary

In [ ]:
# Load an eval log
eval_log_path = '/workspace/rl-character/christine_experiments/20250923_evals/character/character/think_answer_easy/vllm/Qwen2.5-7B-Instruct_sonnet37_hack_0.3_chat_0.3_20000_limitcode_lr5_6/2025-09-24T08-23-10+00-00_judge-character-specialcase_G5sc2mUc2oLn56Xha9Unsk.eval'
evallog = read_eval_log(eval_log_path)

# Extract reasoning for prod_binary (idx=3 for this eval)
reasonings = extract_reasoning_from_evallog(evallog, idx=3, score_key='prod_binary')

print(f"Extracted {len(reasonings)} reasoning instances")
print(f"\nExample reasoning object:")
print(f"  String ID: {reasonings[0].string_id}")
print(f"  Target: {reasonings[0].target}")
print(f"  Score: {reasonings[0].score}")
print(f"  Content length: {len(reasonings[0].content)} chars")

## Step 2: Filter Reasoning (Optional)

You can filter reasoning instances based on target and score.

In [ ]:
# Example: Get only incorrect hack assessments
incorrect_hacks = [r for r in reasonings if r.target == 'hack' and r.score == 'I']
print(f"Found {len(incorrect_hacks)} incorrect hack assessments")

# For this example, let's use a small subset
sample_reasonings = incorrect_hacks[:10]  # Use first 10 for quick testing

## Step 3: Summarize Reasoning with LLM

Use a ReasoningProcessor to add summaries and explanations to each reasoning instance.
This calls an LLM to analyze why the model made an incorrect assessment.

After summarization, each Reasoning object will have:
- `summary`: A summary of why the model failed (configurable number of sentences)
- `explanation`: A detailed explanation of the failure

You can configure the number of sentences in the summary via `num_summary_sentences`.

In [ ]:
# Create a processor
# You can customize: model, temperature, max_concurrent requests, num_summary_sentences
processor = ReasoningProcessor(
    model="claude-sonnet-4-20250514",
    temperature=1.0,
    max_concurrent=5,
    num_summary_sentences=1  # Change this to get longer summaries (e.g., 2, 3)
)

# Summarize the reasoning instances
# This will call the LLM asynchronously for all instances
summarized_reasonings = await processor.summarize_batch(sample_reasonings)

print(f"\nSummarized {len(summarized_reasonings)} reasoning instances")
print(f"\nExample summary:")
print(f"  Summary: {summarized_reasonings[0].summary}")
print(f"\nExample explanation:")
print(f"  {summarized_reasonings[0].explanation[:200]}...")

## Step 4: Save/Load Reasoning

You can save reasoning instances to disk and load them later.

In [ ]:
# Save to JSONL file
save_reasonings(summarized_reasonings, '/tmp/reasonings.jsonl')

# Load from JSONL file
loaded_reasonings = load_reasonings('/tmp/reasonings.jsonl')
print(f"Loaded {len(loaded_reasonings)} reasoning instances")

## Step 5: Cluster Reasoning

Cluster reasoning instances based on their summaries to find patterns.
The LLM will group similar failure modes together.

Each Cluster object contains:
- `cluster_id`: Unique ID for the cluster
- `summary`: Description of what unites this cluster
- `members`: List of Reasoning objects in this cluster

After clustering, each Reasoning object will have a `cluster` attribute pointing to its cluster.

In [ ]:
# Cluster the reasoning instances
# chunk_size controls how many instances are clustered together (default 60)
clusters = await processor.cluster(summarized_reasonings, chunk_size=60)

print(f"\nFound {len(clusters)} clusters")
for i, cluster in enumerate(clusters):
    print(f"\nCluster {i} ({cluster.cluster_id}):")
    print(f"  Summary: {cluster.summary}")
    print(f"  Members: {len(cluster.members)}")

## Step 6: Access Cluster from Reasoning

After clustering, you can access the cluster from any reasoning instance.

In [ ]:
# Pick a reasoning instance
example_reasoning = summarized_reasonings[0]

print(f"Reasoning: {example_reasoning.string_id}")
print(f"Summary: {example_reasoning.summary}")
print(f"\nBelongs to cluster: {example_reasoning.cluster.cluster_id}")
print(f"Cluster summary: {example_reasoning.cluster.summary}")
print(f"Cluster has {len(example_reasoning.cluster.members)} members")

## Step 7: Analyze Clusters

Explore the clusters to understand patterns in model failures.

In [ ]:
# Print detailed cluster analysis
for cluster in clusters:
    print(f"\n{'='*80}")
    print(f"Cluster: {cluster.cluster_id}")
    print(f"Summary: {cluster.summary}")
    print(f"Size: {len(cluster.members)} members")
    print(f"\nExample summaries from this cluster:")
    for i, member in enumerate(cluster.members[:3]):  # Show first 3
        print(f"  {i+1}. {member.summary}")

## Step 8: Save Cluster Information

Save cluster metadata (not including full reasoning content).

In [ ]:
# Save cluster information to JSON
save_clusters(clusters, '/tmp/clusters.json')

# You can also save the reasoning instances with cluster info
save_reasonings(summarized_reasonings, '/tmp/reasonings_with_clusters.jsonl')

## Complete Workflow Example

Here's the full workflow in one cell:

In [ ]:
# 1. Load eval log and extract reasoning
evallog = read_eval_log(eval_log_path)
reasonings = extract_reasoning_from_evallog(evallog, idx=3, score_key='prod_binary')

# 2. Filter (optional)
filtered = [r for r in reasonings if r.target == 'hack' and r.score == 'I']

# 3. Create processor and summarize
processor = ReasoningProcessor(max_concurrent=5, num_summary_sentences=2)
summarized = await processor.summarize_batch(filtered[:20])  # Sample for testing

# 4. Cluster
clusters = await processor.cluster(summarized, chunk_size=60)

# 5. Analyze results
print(f"\nProcessed {len(summarized)} reasonings into {len(clusters)} clusters")
for cluster in clusters:
    print(f"\nCluster {cluster.cluster_id}: {len(cluster.members)} members")
    print(f"  {cluster.summary}")